# A CI gate on real data

`climval check` runs comparisons declared in a configuration file and exits
with 0 when every threshold holds, 1 when a threshold fails and 2 when the
configuration or the data cannot be used. In a CI job the exit code decides
whether the job passes.

This notebook builds such a gate on real open data: ERA5 near-surface air
temperature as the reference, two CMIP6 historical simulations from the
Pangeo catalogue as candidates, over the IPCC AR6 land region Northern Europe
(NEU), 2005 to 2014, by season. It then runs `climval check` exactly as a CI
job would, and shows the Markdown that would appear as the job summary.

About 170 MB are read anonymously from public cloud storage.

In [1]:
import subprocess
import sys
import warnings
from pathlib import Path

import dask
import gcsfs
import numpy as np
import pandas as pd
import regionmask
import xarray as xr
from IPython.display import Markdown, display

import climval

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Consolidated metadata.*")
OUT = Path("output")
OUT.mkdir(exist_ok=True)
fs = gcsfs.GCSFileSystem(token="anon")
print("climval", climval.__version__, "| regionmask", regionmask.__version__)

climval 0.2.0 | regionmask 0.13.0


## The reference: ERA5 monthly means for the region

As in the [Europe example](era5_cmip6_europe.ipynb), ERA5 comes from the
1.5 degree, 6-hourly ARCO-ERA5 store, whose chunks hold two days of the
whole globe. To keep this example small, one two-day chunk in twelve is
read (two days every 24 days, all four analyses of each day), and monthly
means are estimated from the sampled days. The result is written to a small
NetCDF file: in a real CI job this file would be computed once, from the
full record, and committed or cached.

In [2]:
region = climval.MaskRegion(regionmask.defined_regions.ar6.land, "NEU")
box = region.box()
print(region.name, "| bounding box lat", box.lat, "lon", box.lon)

N.Europe | bounding box lat (48.0, 72.6) lon (-10.0, 40.0)


In [3]:
ERA5_STORE = (
    "gcp-public-data-arco-era5/ar/"
    "1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr"
)
era5 = xr.open_zarr("gs://" + ERA5_STORE, storage_options={"token": "anon"}, chunks={})
t2m = era5["2m_temperature"]
step = t2m.encoding["chunks"][0]
times = pd.DatetimeIndex(era5.time.values)
i0 = int(np.searchsorted(times, np.datetime64("2005-01-01")))
i1 = int(np.searchsorted(times, np.datetime64("2015-01-01")))
EVERY = 12
blocks = [k for k in range(-(-i0 // step), i1 // step) if k % EVERY == 0]
index = np.concatenate([np.arange(k * step, (k + 1) * step) for k in blocks])
sizes = {f["name"].rsplit("/", 1)[-1]: f["size"] for f in fs.ls(ERA5_STORE + "/2m_temperature", detail=True)}
era5_bytes = sum(sizes[f"{k}.0.0"] for k in blocks)
print(f"{len(blocks)} chunks, {era5_bytes / 1e6:.0f} MB to read")

152 chunks, 93 MB to read


In [4]:
sub = t2m.isel(time=index)
sub = sub.assign_coords(longitude=((sub.longitude + 180) % 360) - 180).sortby("longitude")
sub = sub.sel(
    latitude=slice(box.lat[0] - 2, box.lat[1] + 2),
    longitude=slice(box.lon[0] - 2, box.lon[1] + 2),
).transpose("time", "latitude", "longitude")
with dask.config.set(scheduler="threads", num_workers=8):
    sub = sub.load()
monthly = sub.resample(time="MS").mean().rename("t2m")
monthly.attrs = {"units": "K", "long_name": "2 m temperature, monthly mean of sampled days"}
reference_file = OUT / "era5_t2m_monthly_neu_2005_2014.nc"
monthly.to_dataset().assign_attrs(
    title="ERA5 2 m temperature, monthly means estimated from sampled days",
    source="ARCO-ERA5, " + ERA5_STORE,
    sampling=f"one 2-day chunk in {EVERY}, all four analyses per day",
    licence="Contains modified Copernicus Climate Change Service information 2026",
).to_netcdf(reference_file)
print(reference_file, f"{reference_file.stat().st_size / 1e3:.0f} kB,", monthly.sizes["time"], "months")

output\era5_t2m_monthly_neu_2005_2014.nc 348 kB, 120 months


## The configuration

Two checks share a `defaults` block: the reference file, the region, the
period, seasonal groups, the ERA5 grid as the comparison grid, and the
thresholds. Each check names one model through a query of the Pangeo CMIP6
catalogue. The limits are set for this demonstration; they are not a
recommendation for what a model should achieve.

In [5]:
config_text = """\
version: 1
defaults:
  reference: {path: era5_t2m_monthly_neu_2005_2014.nc, name: ERA5}
  variable: {reference: t2m, candidate: tas}
  region: "ar6.land:NEU"
  period: ["2005", "2014"]
  by: season
  target: reference
  samples: [space_time, climatology]
  thresholds:
    mean_bias: {abs_max: 1.5, sample: climatology, group: "*"}
    pearson_r: {min: 0.9, sample: climatology, group: "*"}
    crmse: {max: 2.0, sample: climatology, group: "*"}
checks:
  - name: noresm2-lm
    description: NorESM2-LM historical r1i1p1f1 against ERA5, AR6 Northern Europe land
    candidate:
      catalog: https://storage.googleapis.com/cmip6/pangeo-cmip6.json
      query: {source_id: NorESM2-LM, experiment_id: historical, table_id: Amon,
              variable_id: tas, member_id: r1i1p1f1, grid_label: gn}
      storage_options: {token: anon}
      open_kwargs: {consolidated: true}
  - name: mpi-esm1-2-lr
    description: MPI-ESM1-2-LR historical r1i1p1f1 against ERA5, AR6 Northern Europe land
    candidate:
      catalog: https://storage.googleapis.com/cmip6/pangeo-cmip6.json
      query: {source_id: MPI-ESM1-2-LR, experiment_id: historical, table_id: Amon,
              variable_id: tas, member_id: r1i1p1f1, grid_label: gn}
      storage_options: {token: anon}
      open_kwargs: {consolidated: true}
"""
config = OUT / "climval.yml"
config.write_text(config_text, encoding="utf-8")
print(config_text)

version: 1
defaults:
  reference: {path: era5_t2m_monthly_neu_2005_2014.nc, name: ERA5}
  variable: {reference: t2m, candidate: tas}
  region: "ar6.land:NEU"
  period: ["2005", "2014"]
  by: season
  target: reference
  samples: [space_time, climatology]
  thresholds:
    mean_bias: {abs_max: 1.5, sample: climatology, group: "*"}
    pearson_r: {min: 0.9, sample: climatology, group: "*"}
    crmse: {max: 2.0, sample: climatology, group: "*"}
checks:
  - name: noresm2-lm
    description: NorESM2-LM historical r1i1p1f1 against ERA5, AR6 Northern Europe land
    candidate:
      catalog: https://storage.googleapis.com/cmip6/pangeo-cmip6.json
      query: {source_id: NorESM2-LM, experiment_id: historical, table_id: Amon,
              variable_id: tas, member_id: r1i1p1f1, grid_label: gn}
      storage_options: {token: anon}
      open_kwargs: {consolidated: true}
  - name: mpi-esm1-2-lr
    description: MPI-ESM1-2-LR historical r1i1p1f1 against ERA5, AR6 Northern Europe land
    candidate

`--validate` checks the file without opening any data. A typo, an unknown
metric or a threshold on a sample that is not computed would exit with 2
here.

In [6]:
cli = [sys.executable, "-m", "climval.cli"]
done = subprocess.run([*cli, "check", str(config), "--validate"], capture_output=True, text=True)
print(done.stdout, "exit code", done.returncode)

climval check: output\climval.yml is valid (2 checks: noresm2-lm, mpi-esm1-2-lr)
 exit code 0


## Run the gate

This is the command a CI job runs. `--output-dir` writes the JSON result,
the Markdown summary and the HTML report.

In [7]:
done = subprocess.run(
    [*cli, "check", str(config), "--output-dir", str(OUT / "report")],
    capture_output=True,
    text=True,
    timeout=1200,
)
print(done.stdout)
print("exit code", done.returncode)

climval check: running noresm2-lm
climval check: running mpi-esm1-2-lr
  climval check: output\climval.yml
  FAIL   noresm2-lm (t2m/tas)
         reference: era5_t2m_monthly_neu_2005_2014.nc
         candidate: pangeo-cmip6.json [source_id=NorESM2-LM, experiment_id=historical, table_id=Amon, variable_id=tas, member_id=r1i1p1f1, grid_label=gn]
         ok   mean_bias [climatology, all] = -0.3623 (limit abs <= 1.5)
         ok   mean_bias [climatology, DJF] = 0.5303 (limit abs <= 1.5)
         ok   mean_bias [climatology, MAM] = 0.01564 (limit abs <= 1.5)
         ok   mean_bias [climatology, JJA] = -1.316 (limit abs <= 1.5)
         ok   mean_bias [climatology, SON] = -0.6788 (limit abs <= 1.5)
         ok   pearson_r [climatology, all] = 0.9528 (limit >= 0.9)
         ok   pearson_r [climatology, DJF] = 0.9112 (limit >= 0.9)
         ok   pearson_r [climatology, MAM] = 0.9599 (limit >= 0.9)
         ok   pearson_r [climatology, JJA] = 0.9547 (limit >= 0.9)
         ok   pearson_r [clim

## The job summary

With `--github-summary` (or the GitHub Action) the same Markdown is appended
to `$GITHUB_STEP_SUMMARY` and appears on the run's summary page.

In [8]:
display(Markdown((OUT / "report" / "climval-check.md").read_text(encoding="utf-8")))

## climval check

**FAIL: threshold failure (1 of 2 checks passed, 1 failed, 0 errors; exit code 1)**

Configuration: `output\climval.yml` (sha256 `625dde550be1`)

| Status | Check | Variable | Limits passed | Reference | Candidate |
|---|---|---|---|---|---|
| **FAIL** | noresm2-lm | t2m/tas | 14 of 15 | era5_t2m_monthly_neu_2005_2014.nc | pangeo-cmip6.json [source_id=NorESM2-LM, experiment_id=historical, table_id=Amon, variable_id=tas, member_id=r1i1p1f1, grid_label=gn] |
| **PASS** | mpi-esm1-2-lr | t2m/tas | 15 of 15 | era5_t2m_monthly_neu_2005_2014.nc | pangeo-cmip6.json [source_id=MPI-ESM1-2-LR, experiment_id=historical, table_id=Amon, variable_id=tas, member_id=r1i1p1f1, grid_label=gn] |

### FAIL: noresm2-lm (t2m/tas)

NorESM2-LM historical r1i1p1f1 against ERA5, AR6 Northern Europe land

NorESM2-LM r1i1p1f1 against ERA5, variable `t2m` in K.

| Result | Metric | Sample, group | Value | Units | Limit |
|---|---|---|---|---|---|
| **fail** | crmse | climatology, DJF | 2.46 | K | <= 2 |
| pass | mean_bias | climatology, all | -0.3623 | K | abs <= 1.5 |
| pass | mean_bias | climatology, DJF | 0.5303 | K | abs <= 1.5 |
| pass | mean_bias | climatology, MAM | 0.01564 | K | abs <= 1.5 |
| pass | mean_bias | climatology, JJA | -1.316 | K | abs <= 1.5 |
| pass | mean_bias | climatology, SON | -0.6788 | K | abs <= 1.5 |
| pass | pearson_r | climatology, all | 0.9528 | 1 | >= 0.9 |
| pass | pearson_r | climatology, DJF | 0.9112 | 1 | >= 0.9 |
| pass | pearson_r | climatology, MAM | 0.9599 | 1 | >= 0.9 |
| pass | pearson_r | climatology, JJA | 0.9547 | 1 | >= 0.9 |
| pass | pearson_r | climatology, SON | 0.9649 | 1 | >= 0.9 |
| pass | crmse | climatology, all | 1.209 | K | <= 2 |
| pass | crmse | climatology, MAM | 1.219 | K | <= 2 |
| pass | crmse | climatology, JJA | 1.108 | K | <= 2 |
| pass | crmse | climatology, SON | 0.9873 | K | <= 2 |

<details><summary>Alignment steps</summary>

1. inputs: reference_variable=t2m, candidate_variable=tas
1. coordinates: reference={...}, candidate={"grid": "rectilinear", "detected": {"lat": "lat", "lon": "lon", "time": "time"}, "changes": []}
1. target_grid: target=reference, regridded=["candidate"]
1. longitude: frame=[-180.0, 180.0], reference=-180..180 unchanged, candidate=0..360 re-expressed in frame
1. region: region={...}, reference_points={"lat": 17, "lon": 33}, candidate_padded=global longitude: one wrapped column added at each end, candidate_points={"lat": 15, "lon": 23}
1. time: calendars={"reference": "proleptic_gregorian", "candidate": "noleap"}, period=["2005", "2014"], native_frequency={"reference": "monthly", "candidate": "monthly"}, frequency=monthly, matched_periods=120, unmatched_dropped={"reference": 0, "candidate": 0}, method=periods matched by calendar labels (year-month), not by timestamp, range=["2005-01-01T00:00:00.000000000", "2014-12-01T00:00:00.000000000"]
1. regrid: source=candidate, method=bilinear, target_grid={...}, source_grid={...}, target_points_outside_source={"lat": 0, "lon": 0}, target_points_in_source_gaps={"lat": 0, "lon": 0}, action=bilinear interpolation with xarray.interp; target points outside the source grid or inside a gap of it set to NaN
1. region_mask: region=N.Europe, cells_inside=398
1. units: reference_units=K, candidate_units=K, mode=convert, action=units match
1. missing: paired mask: a point counts only where reference and candidate are both valid
1. weights: weights=cos(latitude)

</details>

### PASS: mpi-esm1-2-lr (t2m/tas)

MPI-ESM1-2-LR historical r1i1p1f1 against ERA5, AR6 Northern Europe land

MPI-ESM1-2-LR r1i1p1f1 against ERA5, variable `t2m` in K.

| Result | Metric | Sample, group | Value | Units | Limit |
|---|---|---|---|---|---|
| pass | mean_bias | climatology, all | -0.3068 | K | abs <= 1.5 |
| pass | mean_bias | climatology, DJF | -0.3733 | K | abs <= 1.5 |
| pass | mean_bias | climatology, MAM | 0.05537 | K | abs <= 1.5 |
| pass | mean_bias | climatology, JJA | -0.4105 | K | abs <= 1.5 |
| pass | mean_bias | climatology, SON | -0.4986 | K | abs <= 1.5 |
| pass | pearson_r | climatology, all | 0.9572 | 1 | >= 0.9 |
| pass | pearson_r | climatology, DJF | 0.9823 | 1 | >= 0.9 |
| pass | pearson_r | climatology, MAM | 0.9618 | 1 | >= 0.9 |
| pass | pearson_r | climatology, JJA | 0.948 | 1 | >= 0.9 |
| pass | pearson_r | climatology, SON | 0.9378 | 1 | >= 0.9 |
| pass | crmse | climatology, all | 1.057 | K | <= 2 |
| pass | crmse | climatology, DJF | 1.414 | K | <= 2 |
| pass | crmse | climatology, MAM | 0.9513 | K | <= 2 |
| pass | crmse | climatology, JJA | 1.111 | K | <= 2 |
| pass | crmse | climatology, SON | 1.415 | K | <= 2 |

<details><summary>Alignment steps</summary>

1. inputs: reference_variable=t2m, candidate_variable=tas
1. coordinates: reference={...}, candidate={"grid": "rectilinear", "detected": {"lat": "lat", "lon": "lon", "time": "time"}, "changes": []}
1. target_grid: target=reference, regridded=["candidate"]
1. longitude: frame=[-180.0, 180.0], reference=-180..180 unchanged, candidate=0..360 re-expressed in frame
1. region: region={...}, reference_points={"lat": 17, "lon": 33}, candidate_padded=global longitude: one wrapped column added at each end, candidate_points={"lat": 15, "lon": 29}
1. time: calendars={"reference": "proleptic_gregorian", "candidate": "proleptic_gregorian"}, period=["2005", "2014"], native_frequency={"reference": "monthly", "candidate": "monthly"}, frequency=monthly, matched_periods=120, unmatched_dropped={"reference": 0, "candidate": 0}, method=periods matched by calendar labels (year-month), not by timestamp, range=["2005-01-01T00:00:00.000000000", "2014-12-01T00:00:00.000000000"]
1. regrid: source=candidate, method=bilinear, target_grid={...}, source_grid={...}, target_points_outside_source={"lat": 0, "lon": 0}, target_points_in_source_gaps={"lat": 0, "lon": 0}, action=bilinear interpolation with xarray.interp; target points outside the source grid or inside a gap of it set to NaN
1. region_mask: region=N.Europe, cells_inside=398
1. units: reference_units=K, candidate_units=K, mode=convert, action=units match
1. missing: paired mask: a point counts only where reference and candidate are both valid
1. weights: weights=cos(latitude)

</details>


## Reading the result

The JSON result holds the status, the exit code and, per check, every limit
and the full comparison with its provenance. A later job step, a dashboard
or a comparison with the previous run can read it.

In [9]:
import json

result = json.loads((OUT / "report" / "climval-check.json").read_text(encoding="utf-8"))
print("status", result["status"], "| exit code", result["exit_code"], "| counts", result["counts"])
for check in result["checks"]:
    failed = [f"{t['metric']} [{t['group']}] = {t['value']:.3f}" for t in check["thresholds"] if not t["passed"]]
    print(f"{check['check']}: {check['status']}" + (f"; failed: {', '.join(failed)}" if failed else ""))

status fail | exit code 1 | counts {'pass': 1, 'fail': 1, 'error': 0}
noresm2-lm: fail; failed: crmse [DJF] = 2.460
mpi-esm1-2-lr: pass


In [10]:
rows = [
    {**row, "check": check["check"]}
    for check in result["checks"]
    for row in check["comparison"]["values"]
    if row["sample"] == "climatology" and row["metric"] in ("mean_bias", "pearson_r", "crmse")
]
pd.DataFrame(rows).pivot_table(index=["check", "group"], columns="metric", values="value").round(3)

metric               crmse  mean_bias  pearson_r
check         group                             
mpi-esm1-2-lr DJF    1.414     -0.373      0.982
              JJA    1.111     -0.411      0.948
              MAM    0.951      0.055      0.962
              SON    1.415     -0.499      0.938
              all    1.057     -0.307      0.957
noresm2-lm    DJF    2.460      0.530      0.911
              JJA    1.108     -1.316      0.955
              MAM    1.219      0.016      0.960
              SON    0.987     -0.679      0.965
              all    1.209     -0.362      0.953

In [11]:
steps = result["checks"][0]["comparison"]["provenance"]["steps"]
mask = next(s for s in steps if s["step"] == "region_mask")
print("AR6 NEU land cells on the ERA5 grid:", mask["cells_inside"])

AR6 NEU land cells on the ERA5 grid: 398


The same run from Python, without the command line:

```python
#
report = run_file("output/climval.yml")
report.exit_code, report.counts()
report.outcomes[0].comparison.to_dataframe()
```

## What this shows

The printed console output, the job summary and the table above come from
the same run: each limit, its value and whether it held. The exit code is
what a CI job acts on. With limits like these in a repository, a change to
a model configuration, a post-processing script or the data pipeline that
moves one of these numbers past its limit makes the job fail, and the
summary says which number moved.

The values describe two single-member historical simulations against ERA5
monthly means estimated from sampled days over AR6 Northern Europe land,
2005 to 2014. The sampling adds noise to the ERA5 seasonal means (see the
Europe example for an estimate); for a production gate, compute the
reference from the full record once and commit or cache it.

## In a workflow

```yaml
- uses: northflowlabs/climval@v0.2.0
  with:
    config: climval.yml
    version: "0.2.0"
    extras: cloud,catalog,regions
```

See [the GitHub Action](../how-to/github-action.md) for the inputs and
outputs.

## Data and licences

**ERA5**: Hersbach et al. (2020), https://doi.org/10.1002/qj.3803, read from
ARCO-ERA5 (https://github.com/google-research/arco-era5). Licence to Use
Copernicus Products. Contains modified Copernicus Climate Change Service
information 2026. Neither the European Commission nor ECMWF is responsible
for any use that may be made of the Copernicus information or data it
contains.

**CMIP6**: NorESM2-LM (NCC) and MPI-ESM1-2-LR (MPI-M), historical, r1i1p1f1,
from the Pangeo CMIP6 collection; licensed under CC BY 4.0. The cell below
prints the attributes recorded from each dataset. Eyring et al. (2016),
https://doi.org/10.5194/gmd-9-1937-2016. We acknowledge the World Climate
Research Programme, which, through its Working Group on Coupled Modelling,
coordinated and promoted CMIP6, the climate modelling groups for producing
and making available their model output, the Earth System Grid Federation
for archiving the data and providing access, and the funding agencies who
support CMIP6 and ESGF.

**Region**: IPCC AR6 reference region NEU, Iturbide et al. (2020),
https://doi.org/10.5194/essd-12-2959-2020, through regionmask.

In [12]:
for check in result["checks"]:
    attrs = check["comparison"]["provenance"]["inputs"]["candidate"]["global_attributes"]
    print(check["check"], "|", {k: attrs.get(k) for k in ("source_id", "institution_id", "variant_label", "further_info_url")})

noresm2-lm | {'source_id': 'NorESM2-LM', 'institution_id': 'NCC', 'variant_label': 'r1i1p1f1', 'further_info_url': 'https://furtherinfo.es-doc.org/CMIP6.NCC.NorESM2-LM.historical.none.r1i1p1f1'}
mpi-esm1-2-lr | {'source_id': 'MPI-ESM1-2-LR', 'institution_id': 'MPI-M', 'variant_label': 'r1i1p1f1', 'further_info_url': 'https://furtherinfo.es-doc.org/CMIP6.MPI-M.MPI-ESM1-2-LR.historical.none.r1i1p1f1'}
